In [2]:
pip install scenedetect

  Using cached scenedetect-0.6.7.1-py3-none-any.whl.metadata (3.8 kB)
Using cached scenedetect-0.6.7.1-py3-none-any.whl (130 kB)
Note: you may need to restart the kernel to use updated packages.


In [11]:
import cv2
import os, shutil, zipfile
from typing import List, Tuple
import numpy as np
import pandas as pd
from scenedetect import detect, ContentDetector
import json
import torch
import tempfile
from pathlib import Path


# die müssen vor dem Lauf der Skripte angepasst werden!!!

# Hauptverzeichnis für alle Ergebnisse
SRT_PATH = "srt/Sprechpausen_Bärland.srt"
VIDEO_PATH = "filme/Baerland.mp4" 
HAUPTVERZEICHNIS = Path("extracted_frames")
SUB_VERZEICHNIS = "baer" # Name des Unterprojekts
PROJEKT_PFAD = Path(SUB_VERZEICHNIS) # Basisverzeichnis für dieses Projekt
AUSGABEORDNER_UNFILTERED = HAUPTVERZEICHNIS / SUB_VERZEICHNIS / "unfiltered" # Ausgabeordner wo die extrahierten Bilder ohne Inhalt-Filter gespeichert werden
#AUSGABEORDNER_FILTERED = HAUPTVERZEICHNIS / SUB_VERZEICHNIS / "filtered" # Ausgabeordner wo die extrahierten Bilder nach Inhalt-Filter gespeichert werden
#AUDIOPAUSEN_CSV = HAUPTVERZEICHNIS / SUB_VERZEICHNIS / "audiopausen.csv"
SZENEN_CSV = HAUPTVERZEICHNIS / SUB_VERZEICHNIS / "szenen.csv"
SZENEN_UNFILTERED_CSV = HAUPTVERZEICHNIS / SUB_VERZEICHNIS / "unfiltered.csv" #ungefilterte Bilder
#FRAMES_CSV = AUSGABEORDNER_FILTERED = HAUPTVERZEICHNIS / SUB_VERZEICHNIS / "frames.csv" #gefilterte Bilder

def clean_dir(path):
    """Löscht das angegebene Verzeichnis vollständig nach Benutzerbestätigung und legt es neu an."""
    p = Path(path)
    confirm = input(f"Soll das Verzeichnis '{p}' wirklich GELÖSCHT und neu angelegt werden? (ja/nein): ").strip().lower()
    
    if confirm in ["ja", "j", "yes", "y"]:
        if p.exists():
            shutil.rmtree(p)  # ⚠️ löscht ALLES, auch Unterordner!
        p.mkdir(parents=True, exist_ok=True)
        print(f"✅ Verzeichnis '{p}' wurde neu angelegt.")
    else:
        print("❌ Vorgang abgebrochen – keine Änderungen vorgenommen.")


clean_dir(HAUPTVERZEICHNIS)


# Ordner bei Bedarf automatisch erstellen
AUSGABEORDNER_UNFILTERED.mkdir(parents=True, exist_ok=True)

Soll das Verzeichnis 'extracted_frames' wirklich GELÖSCHT und neu angelegt werden? (ja/nein):  nein


❌ Vorgang abgebrochen – keine Änderungen vorgenommen.


# Bilder-Extraktion
### Einstellung: Szenen ≤ 1.5s bekommen 1 Bild, >1.5s bekommen 3 Bilder

In [12]:
import cv2
import os
import re
from typing import List, Tuple
import numpy as np
from scenedetect import detect, ContentDetector


class MidframeExtractor:
    def __init__(self, 
                 short_scene_threshold: float = 1.5,
                 blur_threshold: float = 100.0,
                 output_dir: str = "extracted_frames"):
        """
        Initialisiert den Midframe Extractor
        
        Args:
            short_scene_threshold: Schwellenwert für kurze vs. lange Szenen in Sekunden
            blur_threshold: Schwellenwert für Unschärfeerkennung (höhere Werte = schärferes Bild erforderlich)
            output_dir: Ausgabeordner für die extrahierten Bilder
        """
        self.short_scene_threshold = short_scene_threshold
        self.blur_threshold = blur_threshold
        self.output_dir = output_dir
        
        # Erstelle Ausgabeordner falls nicht vorhanden
        os.makedirs(self.output_dir, exist_ok=True)
    
    def detect_scene_timestamps(self, video_path: str) -> List[Tuple[float, float]]:
        """
        Erkennt Szenen-Timestamps im Video
        
        Args:
            video_path: Pfad zur Videodatei
            
        Returns:
            Liste von (start_time, end_time) Tupeln in Sekunden
        """
        scene_list = detect(video_path, ContentDetector())
        
        # Konvertiere zu (start, end) Tupeln in Sekunden
        scene_timestamps = []
        for i, scene in enumerate(scene_list):
            start_time = scene[0].get_seconds()
            end_time = scene[1].get_seconds()
            scene_timestamps.append((start_time, end_time))
        
        return scene_timestamps
    
    def calculate_blur_score(self, image: np.ndarray) -> float:
        """
        Berechnet Unschärfe-Score eines Bildes mittels Laplacian Varianz
        
        Args:
            image: BGR Bild als numpy array
            
        Returns:
            Unschärfe-Score (höhere Werte = schärferes Bild)
        """
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        return cv2.Laplacian(gray, cv2.CV_64F).var()
    
    def extract_best_frame_from_timespan(self, cap: cv2.VideoCapture, 
                                       start_time: float, end_time: float,
                                       fps: float, sample_frames: int = 5) -> Tuple[np.ndarray, float]:
        """
        Extrahiert das schärfste Bild aus einem Zeitbereich
        
        Args:
            cap: OpenCV VideoCapture Objekt
            start_time: Startzeit in Sekunden
            end_time: Endzeit in Sekunden
            fps: Frames per Second des Videos
            sample_frames: Anzahl der zu sampelnden Frames für die Auswahl
            
        Returns:
            Tupel aus (best_frame, timestamp_of_best_frame)
        """
        # Berechne Frame-Nummern
        start_frame = int(start_time * fps)
        end_frame = int(end_time * fps)
        
        # Wähle Frames zum Sampeln aus
        total_frames = end_frame - start_frame + 1
        if total_frames <= sample_frames:
            # Wenn wenige Frames, nimm alle
            frames_to_check = list(range(start_frame, end_frame + 1))
        else:
            # Gleichmäßig verteilte Samples
            frames_to_check = np.linspace(start_frame, end_frame, sample_frames, dtype=int)
        
        best_frame = None
        best_blur_score = 0
        best_timestamp = start_time
        
        # Durchlaufe Sample-Frames und finde das schärfste
        for frame_num in frames_to_check:
            if frame_num >= cap.get(cv2.CAP_PROP_FRAME_COUNT):
                break
                
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ret, frame = cap.read()
            
            if not ret:
                continue
                
            blur_score = self.calculate_blur_score(frame)
            
            if blur_score > best_blur_score:
                best_blur_score = blur_score
                best_frame = frame.copy()
                best_timestamp = frame_num / fps
        
        return best_frame, best_timestamp
    
    def extract_frame_at_position(self, cap: cv2.VideoCapture, 
                                start_time: float, end_time: float, 
                                position: str, fps: float) -> Tuple[np.ndarray, float]:
        """
        Extrahiert das beste Frame an einer spezifischen Position (Anfang, Mitte, Ende)
        
        Args:
            cap: OpenCV VideoCapture Objekt
            start_time: Startzeit in Sekunden
            end_time: Endzeit in Sekunden
            position: 'start', 'middle', oder 'end'
            fps: Frames per Second des Videos
            
        Returns:
            Tupel aus (best_frame, timestamp_of_best_frame)
        """
        scene_duration = end_time - start_time
        
        if position == 'start':
            # Erste 20% der Szene oder max 1 Sekunde
            search_end = min(start_time + min(0.2 * scene_duration, 1.0), end_time)
            return self.extract_best_frame_from_timespan(cap, start_time, search_end, fps)
        
        elif position == 'middle':
            # Mittlere 20% der Szene
            middle_point = (start_time + end_time) / 2
            search_radius = min(0.1 * scene_duration, 0.5)  # Max 0.5 Sekunden Radius
            search_start = max(start_time, middle_point - search_radius)
            search_end = min(end_time, middle_point + search_radius)
            return self.extract_best_frame_from_timespan(cap, search_start, search_end, fps)
        
        elif position == 'end':
            # Letzte 20% der Szene oder max 1 Sekunde
            search_start = max(end_time - min(0.2 * scene_duration, 1.0), start_time)
            return self.extract_best_frame_from_timespan(cap, search_start, end_time, fps)
        
        else:
            raise ValueError(f"Unbekannte Position: {position}")
    
    def format_timestamp(self, timestamp: float) -> str:
        """
        Formatiert Timestamp für Dateinamen (HH-MM-SS-mmm)
        
        Args:
            timestamp: Zeit in Sekunden
            
        Returns:
            Formatierter String für Dateinamen
        """
        hours = int(timestamp // 3600)
        minutes = int((timestamp % 3600) // 60)
        seconds = int(timestamp % 60)
        milliseconds = int((timestamp % 1) * 1000)
        
        return f"{hours:02d}-{minutes:02d}-{seconds:02d}-{milliseconds:03d}"
    
    def extract_frames(self, video_path: str, scene_timestamps: List[Tuple[float, float]]) -> List[str]:
            """
            Extrahiert Frames basierend auf Szenen-Timestamps und neuen Regeln.
            ÄNDERUNG: Speichert bei kurzen Szenen das beste Bild auch dann, wenn es unscharf ist.
            """
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                raise ValueError(f"Konnte Video nicht öffnen: {video_path}")
            
            fps = cap.get(cv2.CAP_PROP_FPS)
            video_name = os.path.splitext(os.path.basename(video_path))[0]
            
            extracted_files = []
            
            print(f"Verarbeite {len(scene_timestamps)} Szenen...")
            
            for i, (start_time, end_time) in enumerate(scene_timestamps):
                scene_duration = end_time - start_time
                
                # --- Regel 1: Szenen ≤ 1.5s -> 1 Bild (Mitte der Szene) ---
                if scene_duration <= self.short_scene_threshold:
                    frame, timestamp = self.extract_best_frame_from_timespan(
                        cap, start_time, end_time, fps)
                    
                    if frame is not None:
                        blur_score = self.calculate_blur_score(frame)
                        timestamp_str = self.format_timestamp(timestamp)
                        filename = f"{video_name}_scene{i+1:03d}_mid_{timestamp_str}.jpg"
                        filepath = os.path.join(self.output_dir, filename)
                        
                        # ÄNDERUNG: Wir speichern jetzt IMMER, unterscheiden aber in der Ausgabe
                        if blur_score >= self.blur_threshold:
                            print(f"Szene {i+1}: {scene_duration:.2f}s - 1 Bild extrahiert bei {timestamp:.2f}s (Schärfe: {blur_score:.1f})")
                        else:
                            # Fallback-Meldung
                            print(f"Szene {i+1}: Bild unscharf (Schärfe: {blur_score:.1f} < {self.blur_threshold}), aber als 'bestes verfügbares' gespeichert.")
                        
                        cv2.imwrite(filepath, frame)
                        extracted_files.append(filepath)
                
                # --- Regel 2: Szenen > 1.5s -> 3 Bilder (Anfang, Mitte, Ende) ---
                else:
                    positions = ['start', 'middle', 'end']
                    position_names = ['start', 'mid', 'end']
                    images_extracted = 0
                    
                    for pos, pos_name in zip(positions, position_names):
                        frame, timestamp = self.extract_frame_at_position(
                            cap, start_time, end_time, pos, fps)
                        
                        if frame is not None:
                            blur_score = self.calculate_blur_score(frame)
                            # Hier bleiben wir streng: Nur scharfe Bilder speichern (oder willst du hier auch Fallbacks?)
                            # Aktuell: Streng, aber wir könnten auch hier lockern, wenn gewünscht.
                            if blur_score >= self.blur_threshold:
                                timestamp_str = self.format_timestamp(timestamp)
                                filename = f"{video_name}_scene{i+1:03d}_{pos_name}_{timestamp_str}.jpg"
                                filepath = os.path.join(self.output_dir, filename)
                                
                                cv2.imwrite(filepath, frame)
                                extracted_files.append(filepath)
                                images_extracted += 1
                    
                    print(f"Szene {i+1}: {scene_duration:.2f}s - {images_extracted}/3 Bilder extrahiert")
            
            cap.release()
            return extracted_files
        
    def calculate_scene_statistics(self, scene_timestamps: List[Tuple[float, float]]) -> None:
        """
        Berechnet und zeigt Statistiken über die Szenen an
        
        Args:
            scene_timestamps: Liste von (start_time, end_time) Tupeln
        """
        durations = [end - start for start, end in scene_timestamps]
        
        short_scenes = sum(1 for d in durations if d <= self.short_scene_threshold)
        long_scenes = len(durations) - short_scenes
        
        estimated_images = short_scenes * 1 + long_scenes * 3
        
        print(f"\n=== Szenen-Statistiken ===")
        print(f"Gesamtanzahl Szenen: {len(scene_timestamps)}")
        print(f"Kurze Szenen (≤{self.short_scene_threshold}s): {short_scenes}")
        print(f"Lange Szenen (>{self.short_scene_threshold}s): {long_scenes}")
        print(f"Geschätzte Anzahl Bilder: {estimated_images}")
        print(f"Durchschnittliche Szenenlänge: {np.mean(durations):.2f}s")
        print(f"Median Szenenlänge: {np.median(durations):.2f}s")
    
    def process_video(self, video_path: str) -> List[str]:
        """
        Kompletter Pipeline-Prozess: Szenendetection + Frame-Extraktion
        
        Args:
            video_path: Pfad zur Videodatei
            
        Returns:
            (Liste der Pfade, Liste der Szenen-Timestamps)
        """
        print(f"Starte Szenendetection für: {video_path}")
        scene_timestamps = self.detect_scene_timestamps(video_path)
        
        print(f"Gefundene Szenen: {len(scene_timestamps)}")
        print(f"Einstellungen: short_scene_threshold={self.short_scene_threshold}s, "
              f"blur_threshold={self.blur_threshold}")
        
        # Zeige Statistiken
        self.calculate_scene_statistics(scene_timestamps)
        
        extracted_files = self.extract_frames(video_path, scene_timestamps)
        
        print(f"\nExtraktion abgeschlossen!")
        print(f"Extrahierte Bilder: {len(extracted_files)}")
        print(f"Gespeichert in: {self.output_dir}")
        
        return extracted_files, scene_timestamps


# Szenendetektion + csv

In [13]:
def detect_scene_timestamps (video_path, screenshot_path = None): 
    scene_timestamps = detect(video_path, ContentDetector())
    if screenshot_path != None: 
        videostream = open_video(video_path)
        images = save_images(scene_list=scene_timestamps,video=videostream,output_dir=screenshot_path,num_images=1)
        return (scene_timestamps, images)
    else:
        return scene_timestamps

scene_ts = detect_scene_timestamps(video_path=VIDEO_PATH)
df = pd.DataFrame(scene_ts, columns=["start_s", "end_s"])
df.to_csv(SZENEN_CSV, index=False, encoding="utf-8-sig")
print(f"Szenen (Zeiten) gespeichert in: {SZENEN_CSV}")

Szenen (Zeiten) gespeichert in: extracted_frames/baer/szenen.csv


# Bilder-Extraktion nach Sprechpausen
### Quasi wie ein Lückenfüller - prüft, ob Bilder fehlen, und ggf. nachextrahiert

In [14]:
import re
import os
import cv2

# Hilfsfunktionen (unverändert)
def parse_srt_time(time_str):
    hours, minutes, seconds_ms = time_str.split(':')
    seconds, milliseconds = seconds_ms.split(',')
    return int(hours) * 3600 + int(minutes) * 60 + int(seconds) + int(milliseconds) / 1000.0

def load_pauses_from_srt(srt_path):
    """Liest SRT (tolerant gegenüber Trennzeichen)."""
    pauses = []
    try:
        with open(srt_path, 'r', encoding='utf-8') as f: content = f.read()
    except:
        with open(srt_path, 'r', encoding='latin-1') as f: content = f.read()
    
    pattern = re.compile(r'(\d{2}:\d{2}:\d{2},\d{3})\s+(?:-|-->)\s+(\d{2}:\d{2}:\d{2},\d{3})')
    matches = pattern.findall(content)
    for start_str, end_str in matches:
        start = parse_srt_time(start_str)
        end = parse_srt_time(end_str)
        pauses.append((start, end, end - start))
    print(f"{len(pauses)} Sprechpausen aus SRT geladen.")
    return pauses

def get_timestamp_from_filename(filename):
    match = re.search(r'_(\d{2})-(\d{2})-(\d{2})-(\d{3})\.jpg$', filename)
    if match:
        h, m, s, ms = map(int, match.groups())
        return h * 3600 + m * 60 + s + ms / 1000.0
    return None

def run_gap_filler(extractor, video_path, srt_path, existing_image_paths, scene_timestamps):
    """
    Füllt Lücken basierend auf Audio-Pausen und ordnet sie der korrekten Szene zu.
    """
    print("\n--- Starte Gap-Filler (Pausen-Logik) ---")
    
    pauses = load_pauses_from_srt(srt_path)
    
    # Vorhandene Timestamps sammeln
    existing_timestamps = []
    for p in existing_image_paths:
        ts = get_timestamp_from_filename(os.path.basename(p))
        if ts is not None: existing_timestamps.append(ts)
    existing_timestamps.sort()
    
    # Blöcke berechnen (Aufsummier-Logik)
    required_blocks = []
    block_start = None
    for start, end, duration in pauses:
        if block_start is None: block_start = start
        if duration >= 1.5:
            required_blocks.append({
                'block_start': block_start, 'block_end': end,
                'trigger_pause_center': start + (duration / 2)
            })
            block_start = None
            
    print(f"{len(required_blocks)} relevante Pausen-Blöcke identifiziert.")
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    nachextrahiert = 0
    
    for block in required_blocks:
        b_start, b_end = block['block_start'], block['block_end']
        
        # Check: Gibt es ein Bild?
        if not any((b_start - 0.1) <= ts <= (b_end + 0.1) for ts in existing_timestamps):
            fallback_time = block['trigger_pause_center']
            print(f"Lücke {b_start:.2f}s-{b_end:.2f}s -> Erstelle Fallback bei {fallback_time:.2f}s")
            
            frame, timestamp = extractor.extract_best_frame_from_timespan(
                cap, fallback_time - 0.5, fallback_time + 0.5, fps
            )
            
            if frame is not None:
                # --- SZENEN-ZUORDNUNG ---
                scene_num = 0
                # Wir suchen die Szene, in die dieser Timestamp fällt
                for i, (s_start, s_end) in enumerate(scene_timestamps):
                    if s_start <= timestamp <= s_end + 0.2: # +0.2 Toleranz am Ende
                        scene_num = i + 1
                        break
                
                # Falls Timestamp außerhalb aller Szenen (unwahrscheinlich, aber sicher ist sicher)
                if scene_num == 0:
                    # Versuche, die nächste Szene zu nehmen oder die letzte
                    if timestamp > scene_timestamps[-1][1]: scene_num = len(scene_timestamps)
                    else: scene_num = 999 # Notfall-Nummer
                
                ts_str = extractor.format_timestamp(timestamp)
                
                # Muster: Name_sceneXXX_fallback_Zeit.jpg
                filename = f"{video_name}_scene{scene_num:03d}_fallback_{ts_str}.jpg"
                filepath = os.path.join(extractor.output_dir, filename)
                
                cv2.imwrite(filepath, frame)
                existing_timestamps.append(timestamp)
                existing_image_paths.append(filepath)
                nachextrahiert += 1
            else:
                print("Konnte kein Fallback-Bild extrahieren.")
                
    cap.release()
    print(f"Gap-Filler abgeschlossen. {nachextrahiert} Bilder nachträglich generiert.")
    return existing_image_paths

## Bilder-Extraktion ausführen lassen

In [15]:
if __name__ == "__main__":
    # Flexibel konfigurierbar
    extractor = MidframeExtractor(
        short_scene_threshold=1.5,       # Szenen ≤ 1.5s bekommen 1 Bild, >1.5s bekommen 3 Bilder
        blur_threshold=30.0,            # Mindest-Schärfe-Score WICHTIGER PARAMETER -> hat Einfluss auf die QUalität der extrahirten Bilder
        output_dir=AUSGABEORDNER_UNFILTERED    # Ausgabeordner
    )
    
    # Video verarbeiten
    video_path = VIDEO_PATH  # Pfad zu der Videodatei
    print("=== SCHRITT 1: Visuelle Szenen-Extraktion ===")
    # 1. Normale Szenen-Extraktion (Visueller Pass)
    extracted_files, scene_timestamps = extractor.process_video(video_path)

    print("\n=== SCHRITT 2: Audio-Lückenfüller (Gap-Filler) ===")
    # 2. Gap-Filler (Audio Pass)
    if os.path.exists(SRT_PATH):
        extracted_files = run_gap_filler(extractor, VIDEO_PATH, SRT_PATH, extracted_files, scene_timestamps)
    else:
        print(f"SRT-Datei nicht gefunden: {SRT_PATH}. Gap-Filler übersprungen.")
    
    print(f"\nExtrahierte Dateien:")
    for file in extracted_files[:10]:  # Zeige erste 10 als Beispiel
        print(f"  {file}")
    if len(extracted_files) > 10:
        print(f"  ... und {len(extracted_files) - 10} weitere")
    print(f"\n=== ERGEBNIS: {len(extracted_files)} Bilder gesamt ===")

=== SCHRITT 1: Visuelle Szenen-Extraktion ===
Starte Szenendetection für: filme/Baerland.mp4
Gefundene Szenen: 391
Einstellungen: short_scene_threshold=1.5s, blur_threshold=30.0

=== Szenen-Statistiken ===
Gesamtanzahl Szenen: 391
Kurze Szenen (≤1.5s): 25
Lange Szenen (>1.5s): 366
Geschätzte Anzahl Bilder: 1123
Durchschnittliche Szenenlänge: 6.63s
Median Szenenlänge: 5.00s
Verarbeite 391 Szenen...
Szene 1: 3.88s - 0/3 Bilder extrahiert
Szene 2: 3.80s - 3/3 Bilder extrahiert
Szene 3: 1.88s - 1/3 Bilder extrahiert
Szene 4: 2.40s - 3/3 Bilder extrahiert
Szene 5: 2.84s - 3/3 Bilder extrahiert
Szene 6: 1.36s - 1 Bild extrahiert bei 16.16s (Schärfe: 43.2)
Szene 7: 2.16s - 3/3 Bilder extrahiert
Szene 8: Bild unscharf (Schärfe: 22.7 < 30.0), aber als 'bestes verfügbares' gespeichert.
Szene 9: Bild unscharf (Schärfe: 24.5 < 30.0), aber als 'bestes verfügbares' gespeichert.
Szene 10: 1.24s - 1 Bild extrahiert bei 22.20s (Schärfe: 52.1)
Szene 11: 1.92s - 3/3 Bilder extrahiert
Szene 12: 2.28s - 3/

## CSV erstellen

In [16]:
import os
import csv
import re
from pathlib import Path

def extract_info_from_filename(filename):
    """
    Extrahiert Informationen aus dem Dateinamen.
    Erwartet Format: Name_sceneXXX_wann_timestamp.jpg
    """
    # Entferne die .jpg Endung
    base_name = filename.replace('.jpg', '')
    
    # Regex Pattern für die verschiedenen Teile
    # Pattern erklärt: (.+?)_(scene\d+)_(\w+)_(.+)
    # (.+?) = Name (non-greedy)
    # (scene\d+) = scene + Zahlen
    # (\w+) = wann (Wörter)
    # (.+) = timestamp (Rest)
    pattern = r'(.+?)_(scene\d+)_(\w+)_(.+)'
    
    match = re.match(pattern, base_name)
    
    if match:
        name, scene, wann, timestamp = match.groups()
        return {
            'dateiname': filename,
            'szenennummer': scene,
            'wann': wann,
            'timestamp': timestamp
        }
    else:
        # Falls das Pattern nicht passt, trotzdem Dateiname erfassen
        return {
            'dateiname': filename,
            'szenennummer': 'unbekannt',
            'wann': 'unbekannt',
            'timestamp': 'unbekannt'
        }

def create_csv_from_images(directory_path, output_csv='hamster_bilder.csv'):
    """
    Erstellt eine CSV-Datei aus allen JPG-Dateien im angegebenen Verzeichnis.
    Ignoriert Unterordner.
    """
    directory = Path(directory_path)
    
    if not directory.exists():
        print(f"Fehler: Verzeichnis '{directory_path}' existiert nicht.")
        return
    
    # Sammle alle .jpg Dateien (nur im Hauptverzeichnis, keine Unterordner)
    jpg_files = [f for f in directory.iterdir() if f.is_file() and f.suffix.lower() == '.jpg']
    
    if not jpg_files:
        print(f"Keine JPG-Dateien im Verzeichnis '{directory_path}' gefunden.")
        return
    
    print(f"{len(jpg_files)} JPG-Dateien gefunden.")
    
    # Erstelle CSV
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['Dateiname', 'Szenennummer', 'wann', 'timestamp']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        # Schreibe Header
        writer.writeheader()
        
        # Verarbeite jede JPG-Datei
        for jpg_file in sorted(jpg_files):
            info = extract_info_from_filename(jpg_file.name)
            
            writer.writerow({
                'Dateiname': info['dateiname'],
                'Szenennummer': info['szenennummer'],
                'wann': info['wann'],
                'timestamp': info['timestamp']
            })
    
    print(f"CSV-Datei '{output_csv}' wurde erfolgreich erstellt.")
    print(f"Verarbeitete Dateien: {len(jpg_files)}")


In [17]:
verzeichnis = AUSGABEORDNER_UNFILTERED # von welchen Bildern soll die csv erstellt werden -> entweder AUSGABEORDNER_UNFILTERED oder AUSGABEORDNER_FILTERED
output_datei = SZENEN_UNFILTERED_CSV # Name für die csv-Datei

print(f"Verarbeite JPG-Dateien aus: {verzeichnis}")
print(f"Ausgabe CSV: {output_datei}")
print("-" * 50)

create_csv_from_images(verzeichnis, output_datei)

# Zeige ein paar Beispielzeilen der erstellten CSV
if os.path.exists(output_datei):
    print("\nErste 5 Zeilen der CSV:")
    print("-" * 50)
    with open(output_datei, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i < 6:  # Header + 5 Zeilen
                print(line.strip())
            else:
                break

Verarbeite JPG-Dateien aus: extracted_frames/baer/unfiltered
Ausgabe CSV: extracted_frames/baer/unfiltered.csv
--------------------------------------------------
1024 JPG-Dateien gefunden.
CSV-Datei 'extracted_frames/baer/unfiltered.csv' wurde erfolgreich erstellt.
Verarbeitete Dateien: 1024

Erste 5 Zeilen der CSV:
--------------------------------------------------
Dateiname,Szenennummer,wann,timestamp
Baerland_scene001_fallback_00-00-01-199.jpg,scene001,fallback,00-00-01-199
Baerland_scene002_end_00-00-06-919.jpg,scene002,end,00-00-06-919
Baerland_scene002_mid_00-00-06-139.jpg,scene002,mid,00-00-06-139
Baerland_scene002_start_00-00-04-620.jpg,scene002,start,00-00-04-620
Baerland_scene003_end_00-00-09-560.jpg,scene003,end,00-00-09-560


# ZIP-File erzeugen

In [18]:
from datetime import datetime

# aktuelles Datum im Format
date = datetime.now().strftime("%Y%m%d_%H%M")
ZIPFILE = f"zips/{date}_{SUB_VERZEICHNIS}.zip"
ZIP_SOURCE = HAUPTVERZEICHNIS / SUB_VERZEICHNIS

import os, zipfile
from pathlib import Path

def zip_folder(folder_path, zip_name):
    folder = Path(folder_path)
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder):
            for file in files:
                file_path = Path(root) / file
                arcname = file_path.relative_to(folder.parent)
                zipf.write(file_path, arcname)
    print(f"Archiv erstellt: {zip_name}")

zip_folder(ZIP_SOURCE, ZIPFILE)

Archiv erstellt: zips/20251122_1723_baer.zip
